In [ ]:
# ==============================================================================
# CONFIGURATION CELL — EDIT THESE VALUES BEFORE RUNNING THE NOTEBOOK
# ==============================================================================

import os

# [Optional] Paste the dataset download link here for reference
DATASET_URL = ""  # e.g. "https://openneuro.org/datasets/ds004584"

# Path to the folder containing your raw EEG (.set) files in Google Drive
DRIVE_DATA_DIR = "/content/drive/MyDrive/Dataset/ds004584-download"

# Directory where per-subject processed feature arrays (.npy) will be saved
EXPORT_DIR = "/content/drive/MyDrive/Dataset/processed_features"

# Base directory where the final merged arrays (X_final.npy, etc.) will be stored
BASE_DIR = "/content/drive/MyDrive/Dataset"

print(f"[CONFIG] Dataset URL : {DATASET_URL or '(not set)'}") 
print(f"[CONFIG] Raw Data    : {DRIVE_DATA_DIR}")
print(f"[CONFIG] Processed   : {EXPORT_DIR}")
print(f"[CONFIG] Base Dir    : {BASE_DIR}")


In [ ]:
# Mount your Google Drive — required to access the dataset and save outputs.
# A browser pop-up will appear asking you to authorise access.
from google.colab import drive
drive.mount('/content/drive')
print("[DRIVE] Google Drive mounted successfully at /content/drive")


In [ ]:
# ==============================================================================
# CELL 1: ENVIRONMENT BOOTSTRAPPER (RUN THIS FIRST)
# ==============================================================================
print("[SYSTEM] Initializing cloud container environment...")

# 1. Install MNE-Python for neuroimaging ingestion and filtering
!pip install -q mne

# 2. Install nolds and antropy to unlock chaotic feature metrics
!pip install -q nolds antropy

import os
import mne
import numpy as np

print("[SUCCESS] All neuroimaging packages initialized. You can now run your pipeline cells safely.")

In [ ]:
import os
import mne

# Assign your verified Google Drive path

def verify_drive_dataset(data_dir):
    print("=" * 60)
    print(f"[RUNNING] Testing Path: {data_dir}")
    print("=" * 60)

    # Recursively locate all EEGLAB .set data descriptors
    found_files = []
    for root, _, files in os.walk(data_dir):
        for file in files:
            if file.endswith('.set'):
                found_files.append(os.path.join(root, file))

    print(f"-> Total subject records discovered: {len(found_files)}")

    if len(found_files) == 0:
        print("[FATAL] Zero files detected. Double check if the folder name matches exactly.")
        return False

    try:
        sample_path = sorted(found_files)[0]
        # Ingest metadata structures without loading the heavy binary trace arrays yet
        raw = mne.io.read_raw_eeglab(sample_path, preload=False, verbose=False)
        print(f"\n[INTEGRITY PASSED] Successfully parsed target header properties:")
        print(f"   * Subject File: {os.path.basename(sample_path)}")
        print(f"   * Hardware Channels: {len(raw.ch_names)} (64-channel matrix) ")
        print(f"   * Sampling Frequency: {raw.info['sfreq']} Hz ")
        return True
    except Exception as e:
        print(f"[FATAL] MNE dataset parser failed: {str(e)}")
        return False

# Execute validation pass
verify_drive_dataset(DRIVE_DATA_DIR)

In [ ]:
import os
import mne

# 1. Reuse the verified directory variables

# 2. Grab all subject file paths
found_files = []
for root, _, files in os.walk(DRIVE_DATA_DIR):
    for file in files:
        if file.endswith('.set'):
            found_files.append(os.path.join(root, file))
subject_files = sorted(found_files)

# 3. Load the first subject trace completely into server RAM
print(f"[INFO] Ingesting binary matrix for: {os.path.basename(subject_files[0])}")
raw_signal = mne.io.read_raw_eeglab(subject_files[0], preload=True, verbose=False)

# 4. Filter out standard 60Hz ambient electrical line hum [cite: 368]
print(" -> Applying 60Hz powerline notch filter...")
raw_signal.notch_filter(freqs=60.0, method='iir', verbose=False)

# 5. Extract Stream 1: All Bands (0.1 - 100 Hz bandpass filter)
print(" -> Filtering Stream 1: All Bands (0.1 - 100 Hz)...")
stream_all = raw_signal.copy().filter(
    l_freq=0.1, h_freq=100.0, method='iir',
    iir_params=dict(order=4, ftype='butter', phase='zero'), verbose=False
)

# 6. Extract Stream 2: Alpha Band Only (8 - 12 Hz bandpass filter) [cite: 411]
print(" -> Filtering Stream 2: Alpha Band Only (8 - 12 Hz)...")
stream_alpha = raw_signal.copy().filter(
    l_freq=8.0, h_freq=12.0, method='iir',
    iir_params=dict(order=4, ftype='butter', phase='zero'), verbose=False
)

# 7. Extract Stream 3: Beta Band Only (13 - 30 Hz bandpass filter) [cite: 411]
print(" -> Filtering Stream 3: Beta Band Only (13 - 30 Hz)...")
stream_beta = raw_signal.copy().filter(
    l_freq=13.0, h_freq=30.0, method='iir',
    iir_params=dict(order=4, ftype='butter', phase='zero'), verbose=False
)

print("\n" + "="*50)
print("=== EXPECTED SUB-BAND VERIFICATION ===")
print("=" * 50)
print(f"Stream All Matrix Shape  : {stream_all.get_data().shape}")
print(f"Stream Alpha Matrix Shape: {stream_alpha.get_data().shape}")
print(f"Stream Beta Matrix Shape : {stream_beta.get_data().shape}")
print("="*50)

In [ ]:
import mne
from mne.preprocessing import ICA

print("[PROCESSING] Initializing High-Order Statistical Source Separation...")

# 1. Instantiate the FastICA object using your verified 95% variance parameter
ica = ICA(
    n_components=0.95,
    method='fastica',
    random_state=42,
    max_iter=1000
)

# 2. Fit the ICA unmixing matrix to your Stream All dataset
print(" -> Computing unmixing matrix via Kurtosis maximization (FastICA)...")
ica.fit(stream_all)

# 3. Detect Eye-blink artifacts using the Frontopolar 1 (Fp1) forehead electrode as a proxy
print(" -> Scanning components using forehead channel Fp1 as eye-blink proxy...")
eog_indices, eog_scores = ica.find_bads_eog(
    stream_all,
    ch_name='Fp1',  # Enforces Fp1 as the reference pattern matching eye blinks
    threshold=2.5,
    verbose=False
)

print(f" -> Automated scanner flagged component indices for exclusion: {eog_indices}")

# 4. Zero-out the identified eye-blink vectors from your Alpha and Beta tracks
print("\n[CLEANING] Stripping artifact vectors from target rhythms...")
ica.exclude = eog_indices
clean_alpha = ica.apply(stream_alpha.copy(), verbose=False)
clean_beta  = ica.apply(stream_beta.copy(), verbose=False)

print("\n" + "="*50)
print("=== ARTIFACT REJECTION VERIFICATION ===")
print("="*50)
print(f"Total ICA Components Generated: {ica.n_components_}")
print(f"Components Flagged for Dropping: {ica.exclude}")
print(f"Clean Alpha Output Shape      : {clean_alpha.get_data().shape}")
print(f"Clean Beta Output Shape       : {clean_beta.get_data().shape}")
print("="*50)

In [ ]:
import mne
import numpy as np

print("[PROCESSING] Slicing continuous rhythms into 2-second non-overlapping epochs...")

# 1. Enforce the paper's rigid parameters
fs = 500  # Sampling frequency
segment_samples = 1000  # 2 seconds * 500Hz
required_segments = 20

# Extract the raw numpy matrices from MNE objects
data_alpha = clean_alpha.get_data()
data_beta = clean_beta.get_data()
total_samples = data_alpha.shape[1]

# 2. Extract exactly 20 segments from the stable middle of the recording
# This bypasses initial entry noise or ending boundary cutoff anomalies
start_sample = (total_samples - (required_segments * segment_samples)) // 2
end_sample = start_sample + (required_segments * segment_samples)

epochs_alpha = []
epochs_beta = []

for i in range(required_segments):
    chunk_start = start_sample + (i * segment_samples)
    chunk_end = chunk_start + segment_samples

    # Slice the channels along the column timeline
    slice_a = data_alpha[:, chunk_start:chunk_end]
    slice_b = data_beta[:, chunk_start:chunk_end]

    epochs_alpha.append(slice_a)
    epochs_beta.append(slice_b)

# Convert lists into highly optimized 3D NumPy arrays (Tensors)
# Target Array Dimensions: (Epochs, Channels, Samples)
X_alpha_subject = np.array(epochs_alpha)
X_beta_subject = np.array(epochs_beta)

print("\n" + "="*50)
print("=== SEGMENTATION TENSOR VERIFICATION ===")
print("="*50)
print(f"Alpha Epoch Tensor Shape: {X_alpha_subject.shape}")
print(f"Beta Epoch Tensor Shape : {X_beta_subject.shape}")
print(f"Total Points per Segment: {X_alpha_subject.shape[2]} columns")
print("="*50)

In [ ]:
import numpy as np

def compute_stable_hurst(X):
    """Calculates Hurst Exponent using a fixed, stable aggregate variance scaling loop."""
    N = len(X)
    divs = [1000, 500, 250, 125]
    rs_values = []

    for w in divs:
        num_chunks = N // w
        chunks = X[:num_chunks * w].reshape(num_chunks, w)

        mean_centered = chunks - np.mean(chunks, axis=1, keepdims=True)
        cum_sum = np.cumsum(mean_centered, axis=1)
        R = np.max(cum_sum, axis=1) - np.min(cum_sum, axis=1)
        S = np.std(chunks, axis=1)
        S[S == 0] = 1e-12

        rs_values.append(np.mean(R / S))

    poly = np.polyfit(np.log(divs), np.log(rs_values), 1)
    return np.clip(poly[0], 0.01, 0.99)

def compute_stable_hfd(X, kmax=5):
    """Calculates Higuchi Fractal Dimension using standard interval normalization."""
    N = len(X)
    L = []
    k_vals = np.arange(1, kmax + 1)

    for k in k_vals:
        Lk = []
        for m in range(k):
            sub_seq = X[m::k]
            n_max = len(sub_seq)
            if n_max < 2: continue

            dists = np.sum(np.abs(np.diff(sub_seq)))
            norm_factor = (N - 1) / (np.floor((N - m - 1) / k) * k * k)
            Lk.append(dists * norm_factor)

        if len(Lk) > 0:
            L.append(np.mean(Lk))
        else:
            L.append(1e-12)

    L = np.array(L)
    L[L <= 0] = 1e-12
    poly = np.polyfit(np.log(1.0 / k_vals), np.log(L), 1)
    return np.clip(poly[0], 1.0, 2.0)

def compute_stable_apen(X, m=2, r=0.2):
    """Calculates Approximate Entropy with fixed window shifting slices."""
    N = len(X)
    X_norm = (X - np.mean(X)) / (np.std(X) + 1e-12)

    def _phi(m_dim):
        # FIXED: Both start and end indices shift symmetrically to guarantee uniform row lengths
        X_mat = np.array([X_norm[i : N - m_dim + 1 + i] for i in range(m_dim)]).T
        num_patterns = len(X_mat)
        counts = np.zeros(num_patterns)

        for i in range(num_patterns):
            dists = np.max(np.abs(X_mat - X_mat[i]), axis=1)
            counts[i] = np.sum(dists <= r) / num_patterns

        return np.sum(np.log(counts + 1e-12)) / num_patterns

    return np.abs(_phi(m) - _phi(m + 1))

print("[PROCESSING] Running Normalized Matrix Feature Extraction...")

num_epochs = X_alpha_subject.shape[0]
num_channels = X_alpha_subject.shape[1]

features_alpha_block1 = np.zeros((num_epochs, num_channels, 4))
features_beta_block1 = np.zeros((num_epochs, num_channels, 4))

for epoch in range(num_epochs):
    for ch in range(num_channels):
        sig_a_raw = X_alpha_subject[epoch, ch, :]
        sig_b_raw = X_beta_subject[epoch, ch, :]

        sig_a = (sig_a_raw - np.mean(sig_a_raw)) / (np.std(sig_a_raw) + 1e-12)
        sig_b = (sig_b_raw - np.mean(sig_b_raw)) / (np.std(sig_b_raw) + 1e-12)

        # --- Feature 1: Root Mean Differential (Lyapunov proxy) ---
        features_alpha_block1[epoch, ch, 0] = np.log(np.mean(np.diff(sig_a)**2) + 1e-12)
        features_beta_block1[epoch, ch, 0] = np.log(np.mean(np.diff(sig_b)**2) + 1e-12)

        # --- Feature 2: Hurst Exponent (HE) ---
        features_alpha_block1[epoch, ch, 1] = compute_stable_hurst(sig_a)
        features_beta_block1[epoch, ch, 1] = compute_stable_hurst(sig_b)

        # --- Feature 3: Approximate Entropy (ApEn) ---
        features_alpha_block1[epoch, ch, 2] = compute_stable_apen(sig_a_raw)
        features_beta_block1[epoch, ch, 2] = compute_stable_apen(sig_b_raw)

        # --- Feature 4: Higuchi Fractal Dimension (HFD) ---
        features_alpha_block1[epoch, ch, 3] = compute_stable_hfd(sig_a)
        features_beta_block1[epoch, ch, 3] = compute_stable_hfd(sig_b)

print("\n" + "="*50)
print("=== BLOCK 1 FIXED ACCELERATED VERIFICATION ===")
print("="*50)
print(f"Alpha Feature Tensor Shape: {features_alpha_block1.shape}")
print(f"Beta Feature Tensor Shape : {features_beta_block1.shape}")
print("\n--- Subject Sample Metrics (Epoch 0, Channel 0) ---")
print(f"Lyapunov Volatility Proxy: {features_alpha_block1[0, 0, 0]:.4f}")
print(f"Hurst Value (Alpha)      : {features_alpha_block1[0, 0, 1]:.4f}")
print(f"ApEn Value (Alpha)       : {features_alpha_block1[0, 0, 2]:.4f}")
print(f"Fractal Dim Value (Alpha): {features_alpha_block1[0, 0, 3]:.4f}")
print("="*50)

In [ ]:
import numpy as np

print("[PROCESSING] Extracting Block 2: Higher-Order Statistical Moments (Third to Eighth)...")

# Enforce our tensor configurations
num_epochs = X_alpha_subject.shape[0]
num_channels = X_alpha_subject.shape[1]

# Initialize empty feature matrices: 20 epochs x 63 channels x 6 moments (3rd to 8th)
features_alpha_block2 = np.zeros((num_epochs, num_channels, 6))
features_beta_block2 = np.zeros((num_epochs, num_channels, 6))

for epoch in range(num_epochs):
    for ch in range(num_channels):
        # Isolate the raw signal slices
        sig_a = X_alpha_subject[epoch, ch, :]
        sig_b = X_beta_subject[epoch, ch, :]

        # Mean-center and normalize the traces locally
        sig_a_norm = sig_a - np.mean(sig_a)
        sig_b_norm = sig_b - np.mean(sig_b)

        # Simultaneously compute powers from 3 to 8 across the time horizon
        for idx, power in enumerate(range(3, 9)):
            features_alpha_block2[epoch, ch, idx] = np.mean(sig_a_norm ** power)
            features_beta_block2[epoch, ch, idx] = np.mean(sig_b_norm ** power)

print("\n" + "="*50)
print("=== BLOCK 2 MOMENTS VERIFICATION ===")
print("="*50)
print(f"Alpha Block 2 Tensor Shape: {features_alpha_block2.shape}")
print(f"Beta Block 2 Tensor Shape : {features_beta_block2.shape}")
print("\n--- Subject Sample Metrics (Epoch 0, Channel 0) ---")
print(f"3rd Moment (Alpha - Skew Proxy)    : {features_alpha_block2[0, 0, 0]:.4e}")
print(f"4th Moment (Alpha - Kurtosis Proxy): {features_alpha_block2[0, 0, 1]:.4e}")
print(f"8th Moment (Alpha - Max Order Shape): {features_alpha_block2[0, 0, 5]:.4e}")
print("="*50)

In [ ]:
import numpy as np
from scipy.signal import welch

print("[PROCESSING] Extracting Block 3: High-Resolution Frequency-Domain PSD Features...")

num_epochs = X_alpha_subject.shape[0]
num_channels = X_alpha_subject.shape[1]
fs = 500.0  # Sampling rate

features_alpha_block3 = np.zeros((num_epochs, num_channels, 3))
features_beta_block3 = np.zeros((num_epochs, num_channels, 3))

for epoch in range(num_epochs):
    for ch in range(num_channels):
        sig_a = X_alpha_subject[epoch, ch, :]
        sig_b = X_beta_subject[epoch, ch, :]

        # Enforce nfft=512 to interpolate the spectral grid and clear resolution anomalies
        freqs_a, psd_a = welch(sig_a, fs=fs, nperseg=250, noverlap=125, nfft=512)
        alpha_idx = (freqs_a >= 8.0) & (freqs_a <= 12.0)
        # Ensure absolute positive power constraints to stabilize the centroid division
        f_a, p_a = freqs_a[alpha_idx], np.abs(psd_a[alpha_idx])

        freqs_b, psd_b = welch(sig_b, fs=fs, nperseg=250, noverlap=125, nfft=512)
        beta_idx = (freqs_b >= 13.0) & (freqs_b <= 30.0)
        f_b, p_b = freqs_b[beta_idx], np.abs(psd_b[beta_idx])

        # --- Alpha Metrics Calculation ---
        if len(p_a) > 0 and np.sum(p_a) > 0:
            features_alpha_block3[epoch, ch, 0] = f_a[np.argmax(p_a)]  # Peak Freq
            features_alpha_block3[epoch, ch, 1] = np.sum(f_a * p_a) / np.sum(p_a)  # TRUE Mean Centroid
            features_alpha_block3[epoch, ch, 2] = np.sum(p_a)  # Total Power

        # --- Beta Metrics Calculation ---
        if len(p_b) > 0 and np.sum(p_b) > 0:
            features_beta_block3[epoch, ch, 0] = f_b[np.argmax(p_b)]  # Peak Freq
            features_beta_block3[epoch, ch, 1] = np.sum(f_b * p_b) / np.sum(p_b)  # TRUE Mean Centroid
            features_beta_block3[epoch, ch, 2] = np.sum(p_b)  # Total Power

print("\n" + "="*50)
print("=== BLOCK 3 PSD HIGH-RESOLUTION VERIFICATION ===")
print("="*50)
print(f"Alpha Block 3 Tensor Shape: {features_alpha_block3.shape}")
print(f"Beta Block 3 Tensor Shape : {features_beta_block3.shape}")
print("\n--- Subject Sample Metrics (Epoch 0, Channel 0) ---")
print(f"Alpha Peak Frequency : {features_alpha_block3[0, 0, 0]:.2f} Hz")
print(f"Alpha Mean Frequency : {features_alpha_block3[0, 0, 1]:.2f} Hz")  # Must land inside [8.0, 12.0]
print(f"Alpha Total Power    : {features_alpha_block3[0, 0, 2]:.4e}")
print("="*50)

In [ ]:
import numpy as np

print("[PROCESSING] Constructing Master Multimodal Feature Fusion Matrix...")

num_epochs = features_alpha_block1.shape[0]     # 20
num_channels = features_alpha_block1.shape[1]   # 63

# Initialize the finalized publication-grade 3D Master Array footprint
# Dimensions: 20 Epochs x 63 Channels x 26 Integrated Features
X_master_subject = np.zeros((num_epochs, num_channels, 26))

for epoch in range(num_epochs):
    for ch in range(num_channels):
        # Concatenate Block 1 (4), Block 2 (6), and Block 3 (3) for Alpha Band -> 13 features
        alpha_vector = np.concatenate([
            features_alpha_block1[epoch, ch, :],
            features_alpha_block2[epoch, ch, :],
            features_alpha_block3[epoch, ch, :]
        ])

        # Concatenate Block 1 (4), Block 2 (6), and Block 3 (3) for Beta Band -> 13 features
        beta_vector = np.concatenate([
            features_beta_block1[epoch, ch, :],
            features_beta_block2[epoch, ch, :],
            features_beta_block3[epoch, ch, :]
        ])

        # Fuse both bands side-by-side into the master tensor index allocation
        X_master_subject[epoch, ch, :] = np.concatenate([alpha_vector, beta_vector])

print("\n" + "="*50)
print("=== MASTER PIPELINE MULTIMODAL INTEGRATION ===")
print("="*50)
print(f"Final Master Feature Tensor Shape: {X_master_subject.shape}")
print(f" -> Total Data Nodes Computed   : {X_master_subject.shape[0] * X_master_subject.shape[1]}")
print(f" -> Features Engineered per Node: {X_master_subject.shape[2]} dimensions")
print("="*50)

In [ ]:
# NOTE: This is the original single-threaded batch cell (slow).
# Cell 12 below contains the optimised high-speed version.
# Skip this cell if you have already run Cell 12 or want the faster path.
#
import os
import mne
import numpy as np

# 1. Re-verify the source path and initialize the export directory
os.makedirs(EXPORT_DIR, exist_ok=True)

# 2. Re-compile all subject entries across the directory tree
found_files = []
for root, _, files in os.walk(DRIVE_DATA_DIR):
    for file in files:
        if file.endswith('.set'):
            found_files.append(os.path.join(root, file))
subject_files = sorted(found_files)
total_subjects = len(subject_files)

print(f"[COHORT ENGINE] Starting batch-processing loop for {total_subjects} subjects...")
print(f"[COHORT ENGINE] Final matrices will be exported to: '{EXPORT_DIR}'\n")

# Helper math definitions developed in Phase 3
def compute_stable_hurst(X):
    N = len(X)
    divs = [1000, 500, 250, 125]
    rs_values = []
    for w in divs:
        num_chunks = N // w
        chunks = X[:num_chunks * w].reshape(num_chunks, w)
        mean_centered = chunks - np.mean(chunks, axis=1, keepdims=True)
        cum_sum = np.cumsum(mean_centered, axis=1)
        R = np.max(cum_sum, axis=1) - np.min(cum_sum, axis=1)
        S = np.std(chunks, axis=1)
        S[S == 0] = 1e-12
        rs_values.append(np.mean(R / S))
    poly = np.polyfit(np.log(divs), np.log(rs_values), 1)
    return np.clip(poly[0], 0.01, 0.99)

def compute_stable_hfd(X, kmax=5):
    N = len(X)
    L = []
    k_vals = np.arange(1, kmax + 1)
    for k in k_vals:
        Lk = []
        for m in range(k):
            sub_seq = X[m::k]
            n_max = len(sub_seq)
            if n_max < 2: continue
            dists = np.sum(np.abs(np.diff(sub_seq)))
            norm_factor = (N - 1) / (np.floor((N - m - 1) / k) * k * k)
            Lk.append(dists * norm_factor)
        L.append(np.mean(Lk) if len(Lk) > 0 else 1e-12)
    L = np.array(L)
    L[L <= 0] = 1e-12
    poly = np.polyfit(np.log(1.0 / k_vals), np.log(L), 1)
    return np.clip(poly[0], 1.0, 2.0)

def compute_fully_vectorized_apen(X, m=2, r=0.2):
    N = len(X)
    X_norm = (X - np.mean(X)) / (np.std(X) + 1e-12)
    def _phi(m_dim):
        X_mat = np.array([X_norm[i : N - m_dim + 1 + i] for i in range(m_dim)]).T
        diffs = np.abs(X_mat[:, None, :] - X_mat[None, :, :])
        max_dists = np.max(diffs, axis=2)
        counts = np.sum(max_dists <= r, axis=1) / len(X_mat)
        return np.sum(np.log(counts + 1e-12)) / len(X_mat)
    return np.abs(_phi(m) - _phi(m + 1))

# Global Iteration Loop
for idx, sub_path in enumerate(subject_files):
    sub_name = os.path.basename(sub_path).replace('.set', '')
    target_export_path = os.path.join(EXPORT_DIR, f"{sub_name}_features.npy")

    # Skip processing if this subject was already calculated in a previous run
    if os.path.exists(target_export_path):
        print(f"[{idx+1}/{total_subjects}] Skipping {sub_name} (Already computed).")
        continue

    print(f"[{idx+1}/{total_subjects}] Processing: {sub_name}...")

    try:
        # Load and Notch Filter
        raw_signal = mne.io.read_raw_eeglab(sub_path, preload=True, verbose=False)
        raw_signal.notch_filter(freqs=60.0, method='iir', verbose=False)

        # Enforce dynamic channel check (handles variations across recording setups)
        current_channels = len(raw_signal.ch_names)

        # Sub-band filtration arrays
        stream_alpha = raw_signal.copy().filter(l_freq=8.0, h_freq=12.0, method='iir', iir_params=dict(order=4, ftype='butter', phase='zero'), verbose=False)
        stream_beta = raw_signal.copy().filter(l_freq=13.0, h_freq=30.0, method='iir', iir_params=dict(order=4, ftype='butter', phase='zero'), verbose=False)

        # Epoch segmentation (2-second windows)
        data_alpha = stream_alpha.get_data()
        data_beta = stream_beta.get_data()
        total_samples = data_alpha.shape[1]

        fs, segment_samples, required_segments = 500, 1000, 20
        start_sample = (total_samples - (required_segments * segment_samples)) // 2

        X_alpha = np.array([data_alpha[:, (start_sample + i*segment_samples):(start_sample + (i+1)*segment_samples)] for i in range(required_segments)])
        X_beta = np.array([data_beta[:, (start_sample + i*segment_samples):(start_sample + (i+1)*segment_samples)] for i in range(required_segments)])

        # Feature matrix initial allocation structures
        f_alpha_b1 = np.zeros((required_segments, current_channels, 4))
        f_beta_b1  = np.zeros((required_segments, current_channels, 4))
        f_alpha_b2 = np.zeros((required_segments, current_channels, 6))
        f_beta_b2  = np.zeros((required_segments, current_channels, 6))
        f_alpha_b3 = np.zeros((required_segments, current_channels, 3))
        f_beta_b3  = np.zeros((required_segments, current_channels, 3))

        # Compute multi-domain feature vectors loop
        for epoch in range(required_segments):
            for ch in range(current_channels):
                sig_a_raw, sig_b_raw = X_alpha[epoch, ch, :], X_beta[epoch, ch, :]
                sig_a = (sig_a_raw - np.mean(sig_a_raw)) / (np.std(sig_a_raw) + 1e-12)
                sig_b = (sig_b_raw - np.mean(sig_b_raw)) / (np.std(sig_b_raw) + 1e-12)

                # Block 1 Math
                f_alpha_b1[epoch, ch, 0] = np.log(np.mean(np.diff(sig_a)**2) + 1e-12)
                f_beta_b1[epoch, ch, 0]  = np.log(np.mean(np.diff(sig_b)**2) + 1e-12)
                f_alpha_b1[epoch, ch, 1] = compute_stable_hurst(sig_a)
                f_beta_b1[epoch, ch, 1]  = compute_stable_hurst(sig_b)
                f_alpha_b1[epoch, ch, 2] = compute_fully_vectorized_apen(sig_a_raw)
                f_beta_b1[epoch, ch, 2]  = compute_fully_vectorized_apen(sig_b_raw)
                f_alpha_b1[epoch, ch, 3] = compute_stable_hfd(sig_a)
                f_beta_b1[epoch, ch, 3]  = compute_stable_hfd(sig_b)

                # Block 2 Math
                sa_norm, sb_norm = sig_a_raw - np.mean(sig_a_raw), sig_b_raw - np.mean(sig_b_raw)
                for feat_idx, p in enumerate(range(3, 9)):
                    f_alpha_b2[epoch, ch, feat_idx] = np.mean(sa_norm ** p)
                    f_beta_b2[epoch, ch, feat_idx]  = np.mean(sb_norm ** p)

                # Block 3 Math
                freqs_a, psd_a = mne.time_frequency.psd_array_welch(sig_a_raw, sfreq=fs, fmin=8.0, fmax=12.0, n_fft=512, n_per_seg=250, n_overlap=125, verbose=False)
                freqs_b, psd_b = mne.time_frequency.psd_array_welch(sig_b_raw, sfreq=fs, fmin=13.0, fmax=30.0, n_fft=512, n_per_seg=250, n_overlap=125, verbose=False)

                if len(psd_a) > 0 and np.sum(psd_a) > 0:
                    f_alpha_b3[epoch, ch, 0] = freqs_a[np.argmax(psd_a)]
                    f_alpha_b3[epoch, ch, 1] = np.sum(freqs_a * psd_a) / np.sum(psd_a)
                    f_alpha_b3[epoch, ch, 2] = np.sum(psd_a)
                if len(psd_b) > 0 and np.sum(psd_b) > 0:
                    f_beta_b3[epoch, ch, 0] = freqs_b[np.argmax(psd_b)]
                    f_beta_b3[epoch, ch, 1] = np.sum(freqs_b * psd_b) / np.sum(psd_b)
                    f_beta_b3[epoch, ch, 2] = np.sum(psd_b)

        # Finalize matrix consolidation for this subject
        X_sub = np.zeros((required_segments, current_channels, 26))
        for epoch in range(required_segments):
            for ch in range(current_channels):
                va = np.concatenate([f_alpha_b1[epoch, ch, :], f_alpha_b2[epoch, ch, :], f_alpha_b3[epoch, ch, :]])
                vb = np.concatenate([f_beta_b1[epoch, ch, :], f_beta_b2[epoch, ch, :], f_beta_b3[epoch, ch, :]])
                X_sub[epoch, ch, :] = np.concatenate([va, vb])

        # Save array chunk immediately to Google Drive to protect compute progress
        np.save(target_export_path, X_sub)
        print(f" -> Successfully exported matrix shape: {X_sub.shape}")

    except Exception as e:
        print(f" -> [ERROR] Failed processing subject {sub_name}: {str(e)}")

print("\n[SUCCESS] Cohort feature extraction processing complete.")

In [ ]:
import os
import mne
import numpy as np
from scipy.signal import welch

os.makedirs(EXPORT_DIR, exist_ok=True)

found_files = []
for root, _, files in os.walk(DRIVE_DATA_DIR):
    for file in files:
        if file.endswith('.set'):
            found_files.append(os.path.join(root, file))
subject_files = sorted(found_files)
total_subjects = len(subject_files)

print(f"[COHORT ENGINE] Starting HIGH-SPEED batch-processing loop for {total_subjects} subjects...")

# Optimized math primitives
def compute_stable_hurst(X):
    N = len(X)
    divs = [1000, 500, 250, 125]
    rs_values = []
    for w in divs:
        num_chunks = N // w
        chunks = X[:num_chunks * w].reshape(num_chunks, w)
        mean_centered = chunks - np.mean(chunks, axis=1, keepdims=True)
        cum_sum = np.cumsum(mean_centered, axis=1)
        R = np.max(cum_sum, axis=1) - np.min(cum_sum, axis=1)
        S = np.std(chunks, axis=1)
        S[S == 0] = 1e-12
        rs_values.append(np.mean(R / S))
    poly = np.polyfit(np.log(divs), np.log(rs_values), 1)
    return np.clip(poly[0], 0.01, 0.99)

def compute_stable_hfd(X, kmax=5):
    N = len(X)
    L = []
    k_vals = np.arange(1, kmax + 1)
    for k in k_vals:
        Lk = []
        for m in range(k):
            sub_seq = X[m::k]
            if len(sub_seq) < 2: continue
            dists = np.sum(np.abs(np.diff(sub_seq)))
            norm_factor = (N - 1) / (np.floor((N - m - 1) / k) * k * k)
            Lk.append(dists * norm_factor)
        L.append(np.mean(Lk) if len(Lk) > 0 else 1e-12)
    L = np.array(L)
    L[L <= 0] = 1e-12
    poly = np.polyfit(np.log(1.0 / k_vals), np.log(L), 1)
    return np.clip(poly[0], 1.0, 2.0)

def compute_fully_vectorized_apen(X, m=2, r=0.2):
    N = len(X)
    X_norm = (X - np.mean(X)) / (np.std(X) + 1e-12)
    def _phi(m_dim):
        X_mat = np.array([X_norm[i : N - m_dim + 1 + i] for i in range(m_dim)]).T
        diffs = np.abs(X_mat[:, None, :] - X_mat[None, :, :])
        max_dists = np.max(diffs, axis=2)
        counts = np.sum(max_dists <= r, axis=1) / len(X_mat)
        return np.sum(np.log(counts + 1e-12)) / len(X_mat)
    return np.abs(_phi(m) - _phi(m + 1))

for idx, sub_path in enumerate(subject_files):
    sub_name = os.path.basename(sub_path).replace('.set', '')
    target_export_path = os.path.join(EXPORT_DIR, f"{sub_name}_features.npy")

    # Fault-tolerant skip check
    if os.path.exists(target_export_path):
        print(f"[{idx+1}/{total_subjects}] Skipping {sub_name} (Saved).")
        continue

    print(f"[{idx+1}/{total_subjects}] Processing: {sub_name}...")

    try:
        raw_signal = mne.io.read_raw_eeglab(sub_path, preload=True, verbose=False)
        raw_signal.notch_filter(freqs=60.0, method='iir', verbose=False)
        current_channels = len(raw_signal.ch_names)

        stream_alpha = raw_signal.copy().filter(l_freq=8.0, h_freq=12.0, method='iir', iir_params=dict(order=4, ftype='butter', phase='zero'), verbose=False)
        stream_beta = raw_signal.copy().filter(l_freq=13.0, h_freq=30.0, method='iir', iir_params=dict(order=4, ftype='butter', phase='zero'), verbose=False)

        data_alpha, data_beta = stream_alpha.get_data(), stream_beta.get_data()
        total_samples = data_alpha.shape[1]

        fs, segment_samples, required_segments = 500, 1000, 20
        start_sample = (total_samples - (required_segments * segment_samples)) // 2

        X_alpha = np.array([data_alpha[:, (start_sample + i*segment_samples):(start_sample + (i+1)*segment_samples)] for i in range(required_segments)])
        X_beta = np.array([data_beta[:, (start_sample + i*segment_samples):(start_sample + (i+1)*segment_samples)] for i in range(required_segments)])

        # ==============================================================================
        # CRITICAL ACCELERATION: COMPUTE WELCH PSD FOR ALL CHANNELS & EPOCHS AT ONCE
        # ==============================================================================
        # Shape: (20, 63, 257)
        freqs_a, psds_a = welch(X_alpha, fs=fs, nperseg=250, noverlap=125, nfft=512, axis=-1)
        freqs_b, psds_b = welch(X_beta, fs=fs, nperseg=250, noverlap=125, nfft=512, axis=-1)

        alpha_mask = (freqs_a >= 8.0) & (freqs_a <= 12.0)
        beta_mask = (freqs_b >= 13.0) & (freqs_b <= 30.0)

        f_a, p_a_cube = freqs_a[alpha_mask], np.abs(psds_a[:, :, alpha_mask])
        f_b, p_b_cube = freqs_b[beta_mask], np.abs(psds_b[:, :, beta_mask])

        # Pre-allocate single subject metrics
        f_alpha_b1 = np.zeros((required_segments, current_channels, 4))
        f_beta_b1  = np.zeros((required_segments, current_channels, 4))
        f_alpha_b2 = np.zeros((required_segments, current_channels, 6))
        f_beta_b2  = np.zeros((required_segments, current_channels, 6))
        f_alpha_b3 = np.zeros((required_segments, current_channels, 3))
        f_beta_b3  = np.zeros((required_segments, current_channels, 3))

        for epoch in range(required_segments):
            for ch in range(current_channels):
                sig_a_raw, sig_b_raw = X_alpha[epoch, ch, :], X_beta[epoch, ch, :]
                sig_a = (sig_a_raw - np.mean(sig_a_raw)) / (np.std(sig_a_raw) + 1e-12)
                sig_b = (sig_b_raw - np.mean(sig_b_raw)) / (np.std(sig_b_raw) + 1e-12)

                # Block 1: Chaotic
                f_alpha_b1[epoch, ch, 0] = np.log(np.mean(np.diff(sig_a)**2) + 1e-12)
                f_beta_b1[epoch, ch, 0]  = np.log(np.mean(np.diff(sig_b)**2) + 1e-12)
                f_alpha_b1[epoch, ch, 1] = compute_stable_hurst(sig_a)
                f_beta_b1[epoch, ch, 1]  = compute_stable_hurst(sig_b)
                f_alpha_b1[epoch, ch, 2] = compute_fully_vectorized_apen(sig_a_raw)
                f_beta_b1[epoch, ch, 2]  = compute_fully_vectorized_apen(sig_b_raw)
                f_alpha_b1[epoch, ch, 3] = compute_stable_hfd(sig_a)
                f_beta_b1[epoch, ch, 3]  = compute_stable_hfd(sig_b)

                # Block 2: Higher-Order Moments
                sa_norm, sb_norm = sig_a_raw - np.mean(sig_a_raw), sig_b_raw - np.mean(sig_b_raw)
                for feat_idx, p in enumerate(range(3, 9)):
                    f_alpha_b2[epoch, ch, feat_idx] = np.mean(sa_norm ** p)
                    f_beta_b2[epoch, ch, feat_idx]  = np.mean(sb_norm ** p)

                # Block 3: Fast Array Spectral Assignment
                p_a = p_a_cube[epoch, ch, :]
                p_b = p_b_cube[epoch, ch, :]

                if np.sum(p_a) > 0:
                    f_alpha_b3[epoch, ch, 0] = f_a[np.argmax(p_a)]
                    f_alpha_b3[epoch, ch, 1] = np.sum(f_a * p_a) / np.sum(p_a)
                    f_alpha_b3[epoch, ch, 2] = np.sum(p_a)
                if np.sum(p_b) > 0:
                    f_beta_b3[epoch, ch, 0] = f_b[np.argmax(p_b)]
                    f_beta_b3[epoch, ch, 1] = np.sum(f_b * p_b) / np.sum(p_b)
                    f_beta_b3[epoch, ch, 2] = np.sum(p_b)

        # Feature Concatenation Assembly
        X_sub = np.zeros((required_segments, current_channels, 26))
        for epoch in range(required_segments):
            for ch in range(current_channels):
                va = np.concatenate([f_alpha_b1[epoch, ch, :], f_alpha_b2[epoch, ch, :], f_alpha_b3[epoch, ch, :]])
                vb = np.concatenate([f_beta_b1[epoch, ch, :], f_beta_b2[epoch, ch, :], f_beta_b3[epoch, ch, :]])
                X_sub[epoch, ch, :] = np.concatenate([va, vb])

        np.save(target_export_path, X_sub)

    except Exception as e:
        print(f" -> [ERROR] Failed processing subject {sub_name}: {str(e)}")

print("\n[SUCCESS] Cohort feature extraction processing complete.")

In [ ]:
import os
import numpy as np

PROCESSED_DIR = EXPORT_DIR

print("[AGGREGATOR] Ingesting processed subject tensors from Drive...")

X_list = []
y_list = []
subject_ids = []

# Fetch all generated feature files
feature_files = sorted([f for f in os.listdir(PROCESSED_DIR) if f.endswith('_features.npy')])

# Set the standard baseline channel count from the first subject file
baseline_path = os.path.join(PROCESSED_DIR, feature_files[0])
baseline_matrix = np.load(baseline_path)
STANDARD_CHANNELS = baseline_matrix.shape[1]  # Safely locks at 63 channels

print(f"[AGGREGATOR] Spatial Baseline locked at: {STANDARD_CHANNELS} channels.")

for idx, f_file in enumerate(feature_files):
    matrix_path = os.path.join(PROCESSED_DIR, f_file)
    sub_features = np.load(matrix_path)
    sub_id = f_file.split('_')[0]

    # --- RIGOROUS SPATIAL ALIGNMENT CHECK ---
    # If the subject contains extra channels (like subject 48), trim them to match the baseline
    if sub_features.shape[1] != STANDARD_CHANNELS:
        print(f" -> [ALIGNING] Subject {sub_id} (Index {idx}) has {sub_features.shape[1]} channels. Slicing down to {STANDARD_CHANNELS}...")
        sub_features = sub_features[:, :STANDARD_CHANNELS, :]

    sub_num = int(sub_id.split('-')[1])

    # Assigning Label: 1 for Parkinson's Disease (PD), 0 for Healthy Control (HC)
    if sub_num <= 50:
        label = 1  # PD
    else:
        label = 0  # HC

    X_list.append(sub_features)
    y_list.append(np.full((sub_features.shape[0],), label))
    subject_ids.append(sub_id)

# Stack everything into unified high-speed numpy arrays
X_final = np.vstack(X_list)
y_final = np.concatenate(y_list)

print("\n" + "="*50)
print("=== FINAL COHORT DATASET SHAPE ===")
print("="*50)
print(f"Master Feature Matrix (X) Shape: {X_final.shape}")
print(f"Master Target Vector  (y) Shape: {y_final.shape}")
print(f"Total Cohort Subjects Loaded   : {len(subject_ids)}")
print(f"Total Machine Learning Samples : {X_final.shape[0]} windows")
print(f"Class Distribution             : PD (1) = {np.sum(y_final==1)} epochs | HC (0) = {np.sum(y_final==0)} epochs")
print("="*50)

# Save the unified datasets directly to your Drive home path
np.save(os.path.join(BASE_DIR, "X_final.npy"), X_final)
np.save(os.path.join(BASE_DIR, "y_final.npy"), y_final)
print("[SUCCESS] Master arrays exported. Ready for downstream classification pipelines.")

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler

print("[PIPELINE] Loading consolidated cohort master tensors from Drive...")
X = np.load(os.path.join(BASE_DIR, "X_final.npy"))
y = np.load(os.path.join(BASE_DIR, "y_final.npy"))

# 1. Stratified Split to cleanly manage the 1:2 class imbalance
# Tracks the patient-to-control ratio identically across both splits
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("\n[PIPELINE] Performing Multi-Dimensional Robust Scaling...")
# Reshape to 2D matrix temporarily to scale across the 26 feature profiles uniformly
num_train_samples, num_channels, num_features = X_train_raw.shape
num_test_samples = X_test_raw.shape[0]

X_train_flat = X_train_raw.reshape(-1, num_features)
X_test_flat = X_test_raw.reshape(-1, num_features)

# RobustScaler handles heavy outliers caused by chaotic nonlinear variations
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train_flat)
X_test_scaled = scaler.transform(X_test_flat)

# 2. Reshape to Deep Learning Input Footprint: (Batch, Channels/Sequence, Features)
X_train = X_train_scaled.reshape(num_train_samples, num_channels, num_features)
X_test = X_test_scaled.reshape(num_test_samples, num_channels, num_features)

print("\n" + "="*50)
print("=== FINAL MODEL-READY SPECIFICATIONS ===")
print("="*50)
print(f"X_train Shape (Input Tensor) : {X_train.shape}")
print(f"y_train Shape (Label Tensor) : {y_train.shape}")
print(f"X_test Shape  (Validation)   : {X_test.shape}")
print(f"y_test Shape  (Validation)   : {y_test.shape}")
print(f"\nTraining Set Class Balance   : PD (1) = {np.sum(y_train==1)} | HC (0) = {np.sum(y_train==0)}")
print(f"Validation Set Class Balance : PD (1) = {np.sum(y_test==1)}  | HC (0) = {np.sum(y_test==0)}")
print("="*50)

# Export finalized model inputs
np.save(os.path.join(BASE_DIR, "X_train.npy"), X_train)
np.save(os.path.join(BASE_DIR, "X_test.npy"), X_test)
np.save(os.path.join(BASE_DIR, "y_train.npy"), y_train)
np.save(os.path.join(BASE_DIR, "y_test.npy"), y_test)
print("[SUCCESS] Training sets locked and saved to Drive. Ready for model definition.")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[ENGINE] Initializing Production-Grade KhoshnevisNet on: {device}")

# Load identical dataset matrices
X_train_np = np.load(os.path.join(BASE_DIR, "X_train.npy"))
X_test_np = np.load(os.path.join(BASE_DIR, "X_test.npy"))
y_train_np = np.load(os.path.join(BASE_DIR, "y_train.npy"))
y_test_np = np.load(os.path.join(BASE_DIR, "y_test.npy"))

# Shape transformation to conform to Conv1D layout: (Batch, Features, Channels)
X_train_t = torch.tensor(X_train_np, dtype=torch.float32).transpose(1, 2)
X_test_t = torch.tensor(X_test_np, dtype=torch.float32).transpose(1, 2)
y_train_t = torch.tensor(y_train_np, dtype=torch.float32).unsqueeze(1)
y_test_t = torch.tensor(y_test_np, dtype=torch.float32).unsqueeze(1)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=64, shuffle=False)

class FinalKhoshnevisNet(nn.Module):
    def __init__(self, in_features=26, num_channels=63):
        super(FinalKhoshnevisNet, self).__init__()

        # FIXED: Spatial Channel Normalization to anchor gradient scales
        self.spatial_norm = nn.LayerNorm([in_features, num_channels])

        # Block 1 Conv Primitives
        self.conv1 = nn.Conv1d(in_channels=in_features, out_channels=64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm1d(64)
        self.relu1 = nn.ReLU()
        self.spatial_dropout = nn.Dropout2d(p=0.2) # Drops entire feature maps to prevent co-adaptation

        # Block 2 Conv Primitives
        self.conv2 = nn.Conv1d(in_channels=64, out_channels=128, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm1d(128)
        self.relu2 = nn.ReLU()

        self.pool = nn.MaxPool1d(kernel_size=2)
        self.dropout1 = nn.Dropout(p=0.4)

        # Dimension tracking: floor(63 / 2) = 31
        flattened_dim = 128 * 31

        self.fc1 = nn.Linear(flattened_dim, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.relu3 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, x):
        x = self.spatial_norm(x)
        x = self.spatial_dropout(self.relu1(self.bn1(self.conv1(x))))
        x = self.relu2(self.bn2(self.conv2(x)))
        x = self.pool(x)
        x = self.dropout1(x)

        x = x.view(x.size(0), -1)
        x = self.relu3(self.bn3(self.fc1(x)))
        x = self.dropout2(x)
        return self.fc2(x)

model = FinalKhoshnevisNet().to(device)

# Class weights configuration
pos_weight = torch.tensor([1584.0 / 800.0]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

# FIXED: Reduced learning rate to 0.0001 + added L2 weight decay to penalize exploding weights
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-2)

print("\n[ENGINE] Launching Stable Deep Classification Optimization...")
epochs = 30

for epoch in range(epochs):
    model.train()
    train_loss, correct_train, total_train = 0.0, 0, 0

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()

        # FIXED: Gradient clipping to prevent vanishing/exploding gradient states
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * batch_x.size(0)
        predictions = (torch.sigmoid(logits) >= 0.5).float()
        correct_train += (predictions == batch_y).sum().item()
        total_train += batch_y.size(0)

    model.eval()
    val_loss, correct_val, total_val = 0.0, 0, 0

    with torch.no_grad():
        for batch_x, batch_y in test_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            logits = model(batch_x)
            loss = criterion(logits, batch_y)

            val_loss += loss.item() * batch_x.size(0)
            predictions = (torch.sigmoid(logits) >= 0.5).float()
            correct_val += (predictions == batch_y).sum().item()
            total_val += batch_y.size(0)

    print(f"Epoch [{epoch+1:02d}/{epochs}] -> Train Loss: {train_loss/total_train:.4f} | Train Acc: {(correct_train/total_train)*100:.2f}% || Val Loss: {val_loss/total_val:.4f} | Val Acc: {(correct_val/total_val)*100:.2f}%")

print("\n" + "="*50)
print("=== FINAL CHANNELS NETWORK PIPELINE COMPLETE ===")
print("="*50)

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report
import seaborn as sns

# Ensure the evaluation runs on the same device footprint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.eval()
all_preds = []
all_targets = []

print("[EVALUATOR] Running complete validation set prediction sweep...")

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        probabilities = torch.sigmoid(logits).cpu().numpy()

        all_preds.extend(probabilities)
        all_targets.extend(batch_y.numpy())

all_preds = np.array(all_preds)
all_targets = np.array(all_targets)
binary_preds = (all_preds >= 0.5).astype(float)

# 1. Compute Standard Evaluation Metrics
cm = confusion_matrix(all_targets, binary_preds)
fpr, tpr, thresholds = roc_curve(all_targets, all_preds)
roc_auc = auc(fpr, tpr)

# 2. Plot Consolidated Diagnostic Metrics Window
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

# Plot A: Confusion Matrix Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax[0],
            xticklabels=['Healthy Control (0)', 'Parkinson\'s (1)'],
            yticklabels=['Healthy Control (0)', 'Parkinson\'s (1)'])
ax[0].set_title('Spatial Classifier Confusion Matrix')
ax[0].set_xlabel('Predicted Label')
ax[0].set_ylabel('True Label')

# Plot B: ROC-AUC Curve Track
ax[1].plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
ax[1].plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
ax[1].set_xlim([0.0, 1.0])
ax[1].set_ylim([0.0, 1.05])
ax[1].set_xlabel('False Positive Rate')
ax[1].set_ylabel('True Positive Rate')
ax[1].set_title('Receiver Operating Characteristic (ROC)')
ax[1].legend(loc="lower right")
ax[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

# 3. Print Text Classification Report to Console
print("\n" + "="*60)
print("=== FINAL RECOGNITION PERFORMANCE METRICS ===")
print("="*60)
print(classification_report(all_targets, binary_preds, target_names=['Healthy Control (0)', 'Parkinson\'s (1)']))
print("="*60)